# GO proposal for Cepheids

This notebook is used to display the simulated light curves of Cepheids. This is a new samll simulated dataset as the detrending technique used in the MOCKA (Jannsen+2025) was optimized for gamma Doradus pulsators. 

Data location: `/STER/platodata/PLATOSIM/simulations_PlatoGO_Cepheids`

PlatoSim version: `3.7.0-191-g42fc64d4`

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
# Built-in
import os
import sys
import glob

# PlatoSim defults
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# PlatoSim libraries
import platosim.utilities as ut
import platosim.noise     as ns
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

from IPython.display import display, HTML
display(HTML("<style>.container {width:70% !important; }</style>"))

In [ ]:
# Global paths
path = Path(os.getenv('PLATO_WORKDIR')) / 'cepheids'
vdir = path / 'varsource'
fdir = path / 'figures'
sdir = Path('/STER/platodata/PLATOSIM/simulations_PlatoGO_Cepheids/output')

## Create variable ligth curves

In [ ]:
for i in [4, 6, 14, 22, 42]:
    # Load pulsation modes
    ID = f'{i}'.zfill(9)
    df = pd.read_feather(vdir / 'pulsations' / f'pulsations_{ID}_001.ftr')
    # Generate time series
    dv = pd.DataFrame()
    dv['time'] = np.arange(0, 2*ut.year(), 25) 
    dv['dmag'] = ns.timeSeriesFromFourier(dv.time/86400, df.freq, df.ampl, df.phase)
    # Safe varsource 
    dv.to_csv(vdir / f'varsource_{ID}_001.ftr', index=False, header=False, sep=' ')
    # Plot light curve
    time = dv.time / 86400
    flux = (10**(-0.4*dv.dmag) - 1) * 1e3
    fig = plt.figure(figsize=(9,5))
    plt.plot(time, dv.dmag, 'k-')
    plt.xlabel('Time [days]')
    plt.ylabel('Delta magnitude')
    plt.title(f'Zoom-in for star ID {ID}')
    plt.xlim(0, 30)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    fig.savefig(fdir / f'varsource_{ID}.png', bbox_inches='tight', dpi=200);

---
## Quicklook for raw simulated light curves
---

Generate multi-camera plots:

In [ ]:
for i in [4, 6, 14, 22, 42]:
    # Load light curves
    ID = f'{i}'.zfill(9)
    lcs = LightCurve(sdir / ID, mode="multi")
    # Make plot
    fig, ax = lcs.plot_multi(suffix='hdf5', group=False, camera=False, quarter=False, 
                         flux_median=False, alpha=0.1, figsize=(9,5))
    # Save figure to output
    fig.savefig(fdir / f'lcs_raw_{ID}.png', bbox_inches='tight', dpi=200);

### Star ID 6

In [ ]:
ID = 6
lcs = LightCurve(sdir / f'{ID}'.zfill(9), mode="multi")
len(lcs.files(suffix='hdf5'))

In [ ]:
fig, ax = lcs.plot_multi(suffix='hdf5', group=False, camera=1, quarter=False, 
                         flux_median=144, alpha=0.1, figsize=(9,5))
# ax.set_title(f'Star ID {ID}: N-CAM 1.{c} Q1')
#     fig.savefig(fdir / f'lc_raw_{ID}_ncam1{c}_q1.png', bbox_inches='tight', dpi=200);

In [ ]:
for c in range(1,7):
    fig, ax = lcs.plot_multi(suffix='hdf5', group=False, camera=c, quarter=1, 
                             flux_median=144, alpha=0.1, figsize=(9,5))
    ax.set_title(f'Star ID {ID}: N-CAM 1.{c} Q1')
    fig.savefig(fdir / f'lc_raw_{ID}_ncam1{c}_q1.png', bbox_inches='tight', dpi=200);